In [1]:
%pip install -q -U huggingface_hub

#Kaggle blocks this AWS server's IP (every kaggle.com request gets a 403, even the homepage),
#so kagglehub can't download from here no matter what token is used.
#Instead grab the same iSAID data from Hugging Face:
#   - iSAID annotations (COCO json + instance masks) from isaaccorley/isaid
#   - iSAID images are the DOTA v1.0 images, from isaaccorley/dota
import os
import tarfile
from huggingface_hub import hf_hub_download

path = os.path.expanduser("~/data/isaid")
os.makedirs(path, exist_ok=True)

files = [
    ("isaaccorley/isaid", "isaid_annotations_train.tar.gz"),
    ("isaaccorley/isaid", "isaid_annotations_val.tar.gz"),
    ("isaaccorley/dota", "dotav1.0_images_train.tar.gz"),   #~10 GB
    ("isaaccorley/dota", "dotav1.0_images_val.tar.gz"),     #~3.3 GB
]

for repo, name in files:
    done = os.path.join(path, name + ".done") #marker so re-running the cell skips what's already extracted
    if os.path.exists(done):
        continue
    archive = hf_hub_download(repo_id=repo, filename=name, repo_type="dataset")
    with tarfile.open(archive) as tar:
        tar.extractall(path, filter="data")
    open(done, "w").close()

#Layout: {path}/train|val/images/*.png, {path}/train|val/Annotations/iSAID_*.json, {path}/train|val/Instance_masks/
print("Path to dataset files:", path)


Note: you may need to restart the kernel to use updated packages.
Path to dataset files: /home/ubuntu/data/isaid


In [2]:
import os 

train_path = os.path.join(path, "train")

os.listdir(train_path)








['images', 'Annotations', 'Instance_masks', 'Semantic_masks']

In [ ]:
import os 

train_path = os.path.join(path, "train")

os.listdir(train_path)








['images', 'Annotations', 'Instance_masks', 'Semantic_masks']

In [3]:
train_image_path= os.path.join(train_path, "images")

len(os.listdir(train_image_path))



1411

In [4]:
os.listdir(path)

['train',
 'dotav1.0_images_train.tar.gz.done',
 'isaid_annotations_train.tar.gz.done',
 'dotav1.0_images_val.tar.gz.done',
 'val',
 'isaid_annotations_val.tar.gz.done']

In [5]:
val_path = os.path.join(path, "val")

os.listdir(val_path)

['images', 'Annotations', 'Instance_masks', 'Semantic_masks']

In [6]:
val_images_path = os.path.join(val_path, "images")

len(os.listdir(val_images_path))

458

In [7]:
annotation_path = os.path.join(train_path, "Annotations")

os.listdir(annotation_path)


['iSAID_train.json']

In [8]:
import json 

json_path = os.path.join(annotation_path, 'iSAID_train.json')

with open(json_path) as f:
    coco = json.load(f)

In [9]:
coco.keys()

print(f"Images len:{len(coco["images"])}"), print(f"Annotations len: {len(coco["annotations"])}")

print(f"Categories : {coco["categories"]}")

coco["images"][0], coco["annotations"][0]

Images len:1411
Annotations len: 354789
Categories : [{'id': 1, 'name': 'storage_tank'}, {'id': 2, 'name': 'Large_Vehicle'}, {'id': 3, 'name': 'Small_Vehicle'}, {'id': 4, 'name': 'plane'}, {'id': 5, 'name': 'ship'}, {'id': 6, 'name': 'Swimming_pool'}, {'id': 7, 'name': 'Harbor'}, {'id': 8, 'name': 'tennis_court'}, {'id': 9, 'name': 'Ground_Track_Field'}, {'id': 10, 'name': 'Soccer_ball_field'}, {'id': 11, 'name': 'baseball_diamond'}, {'id': 12, 'name': 'Bridge'}, {'id': 13, 'name': 'basketball_court'}, {'id': 14, 'name': 'Roundabout'}, {'id': 15, 'name': 'Helicopter'}]


({'id': 0,
  'file_name': 'P0000.png',
  'ins_file_name': 'P0000_instance_id_RGB.png',
  'seg_file_name': 'P0000_instance_color_RGB.png'},
 {'id': 0,
  'image_id': 0,
  'segmentation': [[274,
    1602,
    273,
    1603,
    272,
    1603,
    271,
    1603,
    270,
    1604,
    269,
    1604,
    268,
    1604,
    267,
    1604,
    266,
    1605,
    265,
    1605,
    264,
    1605,
    263,
    1606,
    262,
    1606,
    261,
    1606,
    260,
    1607,
    259,
    1607,
    258,
    1607,
    257,
    1608,
    256,
    1608,
    255,
    1608,
    254,
    1609,
    253,
    1610,
    252,
    1611,
    251,
    1611,
    250,
    1612,
    249,
    1613,
    249,
    1614,
    248,
    1615,
    247,
    1616,
    246,
    1617,
    246,
    1618,
    245,
    1619,
    244,
    1620,
    244,
    1621,
    244,
    1622,
    244,
    1623,
    244,
    1624,
    244,
    1625,
    244,
    1626,
    244,
    1627,
    244,
    1628,
    244,
    1629,
    244,
    1630,


In [10]:
from collections import Counter

names = []

for ann in coco["annotations"]:
    names.append(ann["category_name"])  

In [11]:
counts = Counter(names)
    
for name, n in counts.most_common():
    print(name, n)

Small_Vehicle 252828
Large_Vehicle 36505
ship 35597
plane 7980
storage_tank 6088
Harbor 5882
tennis_court 2497
Swimming_pool 2442
Bridge 2078
Helicopter 648
basketball_court 557
baseball_diamond 464
Soccer_ball_field 443
Roundabout 428
Ground_Track_Field 352


In [12]:
from PIL import Image

sizes = []

for img in coco["images"]:
    img_path = os.path.join(train_image_path, img["file_name"])
    
    with Image.open(img_path) as im:
        sizes.append(im.size)
        
        



In [13]:
widths = [s[0] for s in sizes]
heights = [s[1] for s in sizes]



print(f"Sizes: {len(sizes)}")
print(f"Heights: {min(heights)}(min), {max(heights)}(max)")
print(f"Widths: {min(widths)}(min), {max(widths)}(max)")

Sizes: 1411
Heights: 278(min), 8115(max)
Widths: 387(min), 12029(max)
